# ☕ Coffee Bean Sales Intelligence
## A Complete Data Story — 2019 to 2022

---

> *This notebook walks through the full analytics story of a coffee bean retail business — from broad revenue trends down to individual customer behaviour. Each section builds on the last, painting a picture of what's working, what isn't, and where the opportunities lie.*

**Dataset at a glance:**
- 🧾 **1,000 transactions** across four years
- 👤 **913 unique customers** across 3 countries
- ☕ **4 coffee types** × 3 roast levels × 4 pack sizes
- 💰 **\$45,134 total revenue** | **\$4,520 net profit** | 10% avg margin

## 0 · Setup & Data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# ── Global style ──────────────────────────────────────────────────────────────
COFFEE_PALETTE = ['#f0b860', '#e8752a', '#4caf7d', '#5b9bd5', '#9b7fd4', '#e05555', '#4db8b8']
BG      = '#0f0a06'
CARD    = '#1e1509'
GOLD    = '#d4943a'
GOLD2   = '#f0b860'
AMBER   = '#e8752a'
CREAM   = '#f5e6c8'
MUTED   = '#c9a87a'
GREEN   = '#4caf7d'
BLUE    = '#5b9bd5'
RED     = '#e05555'

plt.rcParams.update({
    'figure.facecolor'  : BG,
    'axes.facecolor'    : CARD,
    'axes.edgecolor'    : '#3d2e1a',
    'axes.labelcolor'   : CREAM,
    'axes.titlecolor'   : CREAM,
    'axes.titlesize'    : 14,
    'axes.titleweight'  : 'bold',
    'axes.titlepad'     : 12,
    'xtick.color'       : MUTED,
    'ytick.color'       : MUTED,
    'text.color'        : CREAM,
    'grid.color'        : '#3d2e1a',
    'grid.linewidth'    : 0.6,
    'font.family'       : 'DejaVu Sans',
    'legend.facecolor'  : '#1a1108',
    'legend.edgecolor'  : '#3d2e1a',
    'legend.labelcolor' : CREAM,
})
sns.set_theme(style='dark', rc=plt.rcParams)

print('✅ Libraries loaded and style configured.')

In [ ]:
# ── Reconstructed dataset from dashboard data ──────────────────────────────────

# Monthly revenue 2019-2022
monthly_labels = [
    '2019-01','2019-02','2019-03','2019-04','2019-05','2019-06',
    '2019-07','2019-08','2019-09','2019-10','2019-11','2019-12',
    '2020-01','2020-02','2020-03','2020-04','2020-05','2020-06',
    '2020-07','2020-08','2020-09','2020-10','2020-11','2020-12',
    '2021-01','2021-02','2021-03','2021-04','2021-05','2021-06',
    '2021-07','2021-08','2021-09','2021-10','2021-11','2021-12',
    '2022-01','2022-02','2022-03','2022-04','2022-05','2022-06',
    '2022-07','2022-08'
]
monthly_revenue = [
    828.98,987.40,1021.14,1680.75,398.56,1384.68,1004.14,706.34,1277.02,884.97,823.38,1189.78,
    566.95,1798.34,914.79,761.81,939.36,1438.44,1308.94,300.40,713.05,1514.70,1108.86,751.90,
    837.68,958.83,1544.64,1005.58,907.69,864.53,763.10,1075.91,1643.58,1400.40,1616.18,1147.98,
    1269.42,393.63,1315.20,776.45,1002.37,1155.39,906.73,244.24
]

df_monthly = pd.DataFrame({'period': monthly_labels, 'revenue': monthly_revenue})
df_monthly['date'] = pd.to_datetime(df_monthly['period'])
df_monthly['year'] = df_monthly['date'].dt.year
df_monthly['month'] = df_monthly['date'].dt.month
df_monthly['month_name'] = df_monthly['date'].dt.strftime('%b')

# Annual
df_annual = pd.DataFrame({
    'year'   : [2019, 2020, 2021, 2022],
    'revenue': [12187.16, 12117.54, 13766.11, 7063.44]
})

# Coffee types
df_types = pd.DataFrame({
    'type'  : ['Arabica', 'Excelsa', 'Liberica', 'Robusta'],
    'revenue': [11768.50, 12306.44, 12054.08, 9005.24],
    'profit' : [1059.18, 1353.74, 1566.95, 540.23],
    'weight' : [947, 872, 854, 878],
    'margin' : [9, 11, 13, 6]
})

# Roast
df_roast = pd.DataFrame({
    'roast'  : ['Light', 'Medium', 'Dark'],
    'revenue': [17354.46, 14600.48, 13179.32]
})

# Pack sizes
df_size = pd.DataFrame({
    'size'  : ['0.2 kg', '0.5 kg', '1.0 kg', '2.5 kg'],
    'revenue': [3307.95, 7029.99, 11010.75, 23785.56],
    'profit' : [330.91, 712.27, 1108.30, 2368.61]
})

# Geography
df_country = pd.DataFrame({
    'country': ['United States', 'Ireland', 'United Kingdom'],
    'revenue': [35638.88, 6696.86, 2798.50]
})

df_city = pd.DataFrame({
    'city'   : ['Washington','Houston','Toledo','New York City','Sacramento',
                'Portland','Philadelphia','Dallas','Oklahoma City','Richmond'],
    'revenue': [1066.92, 819.77, 774.18, 772.74, 627.75,
                580.26, 511.24, 505.92, 475.17, 407.59]
})

# Customers
df_customers = pd.DataFrame({
    'name'   : ['Allis Wilmore','Brenn Dundredge','Terri Farra','Nealson Cuttler',
                'Don Flintiff','Derick Snow','Brice Romera','Alexa Sizey',
                'Ailey Brash','Daniel Heinonen'],
    'revenue': [317.07, 307.04, 289.11, 281.68, 278.01,
                251.12, 246.21, 218.73, 206.60, 204.93],
    'orders' : [3, 4, 3, 1, 3, 3, 1, 1, 2, 1]
})

# Loyalty
df_loyalty = pd.DataFrame({
    'status'   : ['No Loyalty Card', 'Loyalty Card'],
    'revenue'  : [24216.40, 20917.85],
    'customers': [470, 443],
    'avg_order': [46.48, 43.67]
})

# Heatmap pivot
hm_data = {
    2019: [828.98,987.40,1021.14,1680.75,398.56,1384.68,1004.14,706.34,1277.02,884.97,823.38,1189.78],
    2020: [566.95,1798.34,914.79,761.81,939.36,1438.44,1308.94,300.40,713.05,1514.70,1108.86,751.90],
    2021: [837.68,958.83,1544.64,1005.58,907.69,864.53,763.10,1075.91,1643.58,1400.40,1616.18,1147.98],
    2022: [1269.42,393.63,1315.20,776.45,1002.37,1155.39,906.73,244.24, None, None, None, None]
}
months = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
df_heatmap = pd.DataFrame(hm_data, index=months).T

print('✅ All datasets ready.')
print(f'   Monthly records : {len(df_monthly)}')
print(f'   Coffee types    : {len(df_types)}')
print(f'   Cities tracked  : {len(df_city)}')
df_monthly.head()

---
## 1 · The Big Picture — KPI Summary

Before diving into charts, let's anchor ourselves with the four headline numbers. These KPIs frame every downstream question: we know the total prize, and now we'll spend the rest of this notebook understanding *how* it was won — and *where* it wasn't.

In [ ]:
kpis = [
    ('💰 Total Revenue',  '$45,134', 'Across all products & regions', GOLD2),
    ('📈 Net Profit',     '$4,520',  'Avg margin: 10.0%',             AMBER),
    ('🧾 Total Orders',   '957',     'Unique transactions',           GREEN),
    ('👤 Customers',      '913',     '479 with loyalty card',         BLUE),
]

fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
fig.patch.set_facecolor(BG)

for ax, (label, value, sub, color) in zip(axes, kpis):
    ax.set_facecolor(CARD)
    for spine in ax.spines.values():
        spine.set_edgecolor('#3d2e1a')
    ax.set_xticks([]); ax.set_yticks([])
    # colour bar on top
    ax.axhline(y=0.97, xmin=0.04, xmax=0.96, linewidth=3, color=color, transform=ax.transAxes)
    ax.text(0.5, 0.70, label, ha='center', va='center', transform=ax.transAxes,
            fontsize=10, color=MUTED, fontweight='bold', letter_spacing=1)
    ax.text(0.5, 0.42, value, ha='center', va='center', transform=ax.transAxes,
            fontsize=28, color='white', fontweight='bold')
    ax.text(0.5, 0.16, sub, ha='center', va='center', transform=ax.transAxes,
            fontsize=9, color=MUTED)

fig.suptitle('☕ Coffee Bean Sales — Key Performance Indicators (2019–2022)',
             fontsize=15, color=CREAM, fontweight='bold', y=1.04)
plt.tight_layout()
plt.show()

**Insight ►** With \$45K revenue and only \$4.5K profit, the average margin sits at 10% — thin but consistent. The real story is in the *mix*: some bean types and sizes are significantly more profitable than others, as we'll discover.

---
## 2 · Sales Performance Over Time

### 2a · Monthly Revenue Trend (2019–2022)

Let's look at how revenue evolved month by month over the four-year window. A time-series view reveals seasonality, anomalies, and the impact of external events.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 5))

x = np.arange(len(df_monthly))
y = df_monthly['revenue'].values

# Area fill
ax.fill_between(x, y, alpha=0.18, color=GOLD2)
ax.plot(x, y, color=GOLD2, linewidth=2.5, zorder=3)
ax.scatter(x, y, color=GOLD2, s=35, zorder=4, edgecolors='#1a1108', linewidths=1.5)

# Year dividers
year_starts = [0, 12, 24, 36]
year_labels = ['2019', '2020', '2021', '2022']
for xs, yl in zip(year_starts, year_labels):
    ax.axvline(xs, color='#3d2e1a', linewidth=1, linestyle='--', alpha=0.7)
    ax.text(xs + 0.3, ax.get_ylim()[1] if ax.get_ylim()[1] > 0 else 1800,
            yl, color=MUTED, fontsize=10, va='top')

# Annotate peak
peak_idx = np.argmax(y)
ax.annotate(f'Peak\n${y[peak_idx]:,.0f}',
            xy=(peak_idx, y[peak_idx]),
            xytext=(peak_idx + 2, y[peak_idx] + 120),
            arrowprops=dict(arrowstyle='->', color=GOLD2, lw=1.5),
            color=GOLD2, fontsize=9, fontweight='bold')

# Annotate trough
trough_idx = np.argmin(y)
ax.annotate(f'Trough\n${y[trough_idx]:,.0f}',
            xy=(trough_idx, y[trough_idx]),
            xytext=(trough_idx + 2, y[trough_idx] - 300),
            arrowprops=dict(arrowstyle='->', color=RED, lw=1.5),
            color=RED, fontsize=9, fontweight='bold')

# X-axis labels — show every 3rd month
tick_positions = x[::3]
tick_labels    = df_monthly['period'].values[::3]
ax.set_xticks(tick_positions)
ax.set_xticklabels(tick_labels, rotation=45, ha='right', fontsize=8)

ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v:,.0f}'))
ax.set_title('Monthly Revenue 2019–2022', fontsize=15, color=CREAM, pad=14)
ax.set_ylabel('Revenue (USD)')
ax.grid(True, axis='y', alpha=0.4)
ax.set_xlim(-0.5, len(x) - 0.5)
plt.tight_layout()
plt.show()

**Insights ►**
- Revenue is **highly volatile** month-to-month, ranging from \$244 to \$1,798 — a 7× swing.
- There is **no clean seasonal pattern** suggesting demand is driven by sporadic bulk orders rather than steady retail consumption.
- The data for 2022 ends in **August**, so the apparent drop in 2022 is a truncation artifact, not a true decline.

### 2b · Annual Revenue Breakdown

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Bar chart ---
ax = axes[0]
colors_annual = [GOLD2, AMBER, GREEN, BLUE]
bars = ax.bar(df_annual['year'].astype(str), df_annual['revenue'],
              color=colors_annual, width=0.5, zorder=3,
              edgecolor='#0f0a06', linewidth=0.8)
ax.bar_label(bars, labels=[f'${v:,.0f}' for v in df_annual['revenue']],
             padding=6, color=CREAM, fontsize=11, fontweight='bold')
ax.set_title('Annual Revenue by Year', fontsize=13)
ax.set_ylabel('Revenue (USD)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v/1000:.0f}K'))
ax.grid(True, axis='y', alpha=0.4)
ax.set_ylim(0, df_annual['revenue'].max() * 1.22)

# --- YoY Growth ---
ax2 = axes[1]
yoy = df_annual['revenue'].pct_change() * 100
yoy_colors = [GREEN if v >= 0 else RED for v in yoy[1:]]
ax2.bar(df_annual['year'].astype(str)[1:], yoy[1:],
        color=yoy_colors, width=0.45, zorder=3, edgecolor='#0f0a06')
ax2.axhline(0, color=MUTED, linewidth=1, linestyle='--')
ax2.set_title('Year-over-Year Revenue Growth (%)', fontsize=13)
ax2.set_ylabel('Growth (%)')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:+.1f}%'))
ax2.grid(True, axis='y', alpha=0.4)
for i, (yr, val) in enumerate(zip(df_annual['year'][1:], yoy[1:])):
    ax2.text(i, val + (1 if val >= 0 else -2),
             f'{val:+.1f}%', ha='center', color=CREAM, fontsize=11, fontweight='bold')

fig.suptitle('Annual Performance Overview', fontsize=15, color=CREAM, fontweight='bold')
plt.tight_layout()
plt.show()

**Insights ►**
- 2019 and 2020 were nearly identical (~\$12.1K each) — the business plateaued early.
- **2021 was the breakout year**, growing **+13.6%** to \$13,766 — the highest annual total.
- 2022 appears to drop sharply, but this is a **data truncation effect** (only 8 months recorded). Annualised, 2022 would project to ~\$10,600.

### 2c · Monthly Sales Heatmap (Year × Month)

A heatmap lets us see *both* time dimensions at once — which year-month combinations drove the most revenue.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 4.5))

# Custom colormap — dark background to amber
from matplotlib.colors import LinearSegmentedColormap
coffee_cmap = LinearSegmentedColormap.from_list(
    'coffee', ['#1a1108', '#5a3010', AMBER, GOLD2], N=256
)

mask = df_heatmap.isnull()
heatmap = sns.heatmap(
    df_heatmap,
    ax=ax,
    cmap=coffee_cmap,
    annot=True,
    fmt='.0f',
    linewidths=2,
    linecolor='#0f0a06',
    mask=mask,
    annot_kws={'size': 8, 'color': '#1a1108', 'fontweight': 'bold'},
    cbar_kws={'label': 'Revenue (USD)', 'shrink': 0.8}
)

# Style colorbar
cbar = heatmap.collections[0].colorbar
cbar.ax.yaxis.set_tick_params(color=MUTED)
plt.setp(cbar.ax.yaxis.get_ticklabels(), color=MUTED)

ax.set_title('Revenue Heatmap — Year × Month (USD)', fontsize=14, color=CREAM, pad=14)
ax.set_xlabel('Month', color=CREAM)
ax.set_ylabel('Year', color=CREAM)
ax.tick_params(colors=MUTED)

# Grey out missing cells
for i, j in zip(*np.where(mask.values)):
    ax.add_patch(plt.Rectangle((j, i), 1, 1, fill=True, color='#1a1108', zorder=3))
    ax.text(j + 0.5, i + 0.5, '—', ha='center', va='center', color='#3d2e1a', fontsize=10)

plt.tight_layout()
plt.show()

**Insights ►**
- **February 2020** was the single highest month (\$1,798) — likely a large bulk order.
- **April 2019** (\$1,681) and **September 2021** (\$1,644) are the next hotspots.
- **August 2022** (\$244) is anomalously low — likely an incomplete month of data.
- No single month consistently dominates across years, confirming the **irregular demand** pattern.

---
## 3 · Product Analysis

### 3a · Revenue & Profit by Coffee Type

We carry four bean varieties. The question isn't just which sells the most — it's which *profits* the most.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

# --- Revenue horizontal bar ---
ax = axes[0]
df_rev_sorted = df_types.sort_values('revenue')
bars = ax.barh(df_rev_sorted['type'], df_rev_sorted['revenue'],
               color=COFFEE_PALETTE[:4], edgecolor='#0f0a06', height=0.55, zorder=3)
ax.bar_label(bars, labels=[f'${v:,.0f}' for v in df_rev_sorted['revenue']],
             padding=6, color=CREAM, fontsize=10)
ax.set_title('Revenue by Coffee Type', fontsize=13)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v/1000:.0f}K'))
ax.grid(True, axis='x', alpha=0.4)
ax.set_xlim(0, df_types['revenue'].max() * 1.25)

# --- Profit doughnut ---
ax2 = axes[1]
wedges, texts, autotexts = ax2.pie(
    df_types['profit'],
    labels=df_types['type'],
    colors=COFFEE_PALETTE[:4],
    autopct='%1.1f%%',
    startangle=140,
    wedgeprops=dict(width=0.55, edgecolor='#0f0a06', linewidth=2),
    pctdistance=0.78
)
for t in texts:      t.set_color(CREAM)
for at in autotexts: at.set_color('#1a1108'); at.set_fontweight('bold'); at.set_fontsize(9)
ax2.set_title('Profit Share by Coffee Type', fontsize=13)
# Centre label
ax2.text(0, 0, '$4,520\nTotal\nProfit', ha='center', va='center',
         color=CREAM, fontsize=10, fontweight='bold')

# --- Margin bar ---
ax3 = axes[2]
df_margin = df_types.sort_values('margin')
bar_colors = [GREEN if v >= 10 else AMBER for v in df_margin['margin']]
bars3 = ax3.barh(df_margin['type'], df_margin['margin'],
                 color=bar_colors, edgecolor='#0f0a06', height=0.55, zorder=3)
ax3.bar_label(bars3, labels=[f'{v}%' for v in df_margin['margin']],
              padding=6, color=CREAM, fontsize=11, fontweight='bold')
ax3.axvline(10, color=MUTED, linewidth=1, linestyle='--', alpha=0.6)
ax3.text(10.2, -0.6, 'Avg 10%', color=MUTED, fontsize=8)
ax3.set_title('Profit Margin % by Bean Type', fontsize=13)
ax3.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:.0f}%'))
ax3.grid(True, axis='x', alpha=0.4)
ax3.set_xlim(0, 18)

fig.suptitle('Coffee Type — Revenue, Profit & Margin Analysis', fontsize=15,
             color=CREAM, fontweight='bold')
plt.tight_layout()
plt.show()

**Insights ►**
- **Excelsa leads on revenue** (\$12,306) and **Liberica leads on profit** (\$1,567 / 13% margin).
- **Robusta is the weakest performer** — lowest revenue (\$9,005) and the thinnest margin (6%). It may warrant a price review or promotional push.
- **Arabica** is a volume seller (high revenue) but only a 9% margin — strong sales don't automatically mean strong profits.

### 3b · Roast Level Preferences

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Pie
ax = axes[0]
roast_colors = [GOLD2, GOLD, AMBER]
wedges, texts, autotexts = ax.pie(
    df_roast['revenue'],
    labels=df_roast['roast'],
    colors=roast_colors,
    autopct='%1.1f%%',
    startangle=90,
    wedgeprops=dict(edgecolor='#0f0a06', linewidth=2),
    explode=[0.05, 0, 0]
)
for t in texts:      t.set_color(CREAM); t.set_fontweight('bold')
for at in autotexts: at.set_color('#1a1108'); at.set_fontweight('bold')
ax.set_title('Revenue Share by Roast Level', fontsize=13)

# Sorted bar
ax2 = axes[1]
df_roast_s = df_roast.sort_values('revenue', ascending=True)
bars = ax2.barh(df_roast_s['roast'], df_roast_s['revenue'],
               color=roast_colors[::-1], height=0.45, zorder=3, edgecolor='#0f0a06')
ax2.bar_label(bars, labels=[f'${v:,.0f}' for v in df_roast_s['revenue']],
              padding=6, color=CREAM, fontsize=11, fontweight='bold')
ax2.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v/1000:.0f}K'))
ax2.set_title('Revenue by Roast Level', fontsize=13)
ax2.grid(True, axis='x', alpha=0.4)
ax2.set_xlim(0, df_roast['revenue'].max() * 1.28)

fig.suptitle('Roast Level Analysis — Customer Preference by Sales Volume',
             fontsize=14, color=CREAM, fontweight='bold')
plt.tight_layout()
plt.show()

**Insights ►**
- **Light roast dominates** at 38.5% of revenue (\$17,354) — customers clearly prefer lighter, fruitier profiles.
- **Dark roast is the least popular** (29.2%) — typically associated with espresso blends, which may not be the primary use case for this customer base.
- A strategic move could be to promote **light roast bundles** or introduce light-roast loyalty promotions.

### 3c · Pack Size — Revenue & Profitability

Pack size is a key pricing lever. Let's see where the money actually comes from.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x = np.arange(len(df_size))
width = 0.38

# Grouped bar
ax = axes[0]
b1 = ax.bar(x - width/2, df_size['revenue'], width, label='Revenue', color=GOLD2,
            zorder=3, edgecolor='#0f0a06', borderradius=4)
b2 = ax.bar(x + width/2, df_size['profit'],  width, label='Profit',  color=GREEN,
            zorder=3, edgecolor='#0f0a06')
ax.set_xticks(x)
ax.set_xticklabels(df_size['size'])
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v/1000:.0f}K'))
ax.set_title('Revenue vs Profit by Pack Size', fontsize=13)
ax.legend()
ax.grid(True, axis='y', alpha=0.4)
ax.bar_label(b1, labels=[f'${v/1000:.1f}K' for v in df_size['revenue']], padding=4, color=CREAM, fontsize=8)
ax.bar_label(b2, labels=[f'${v:.0f}' for v in df_size['profit']], padding=4, color=CREAM, fontsize=8)

# Margin rate
ax2 = axes[1]
margins = (df_size['profit'] / df_size['revenue'] * 100).round(1)
bar_colors = [GREEN if m >= 10 else AMBER for m in margins]
bars = ax2.bar(df_size['size'], margins, color=bar_colors, zorder=3,
               edgecolor='#0f0a06', width=0.45)
ax2.bar_label(bars, labels=[f'{v}%' for v in margins],
              padding=5, color=CREAM, fontsize=12, fontweight='bold')
ax2.axhline(10, color=MUTED, linewidth=1.2, linestyle='--', alpha=0.7)
ax2.text(3.55, 10.3, 'Avg margin', color=MUTED, fontsize=8)
ax2.set_title('Profit Margin % by Pack Size', fontsize=13)
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:.0f}%'))
ax2.grid(True, axis='y', alpha=0.4)
ax2.set_ylim(0, 16)

fig.suptitle('Pack Size — Revenue, Profit & Margin Breakdown',
             fontsize=14, color=CREAM, fontweight='bold')
plt.tight_layout()
plt.show()

**Insights ►**
- **2.5 kg packs generate 53% of total revenue** (\$23,786) — customers buying in bulk are the backbone of this business.
- All pack sizes carry **roughly equal margins** (~10%), meaning size-based pricing is well-calibrated.
- **0.2 kg packs contribute very little** (\$3,308) — they may serve as a trial/sample product but aren't profit drivers. Consider bundling them with larger sizes.

### 3d · Weight Sold by Coffee Type

Revenue tells us value. Units (weight) tell us volume. Together they reveal pricing efficiency.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Volume bar
ax = axes[0]
df_vol = df_types.sort_values('weight', ascending=True)
bars = ax.barh(df_vol['type'], df_vol['weight'],
               color=COFFEE_PALETTE[:4], height=0.5, zorder=3, edgecolor='#0f0a06')
ax.bar_label(bars, labels=[f'{v} units' for v in df_vol['weight']],
             padding=6, color=CREAM, fontsize=10)
ax.set_title('Weight Sold (Units) by Coffee Type', fontsize=13)
ax.grid(True, axis='x', alpha=0.4)
ax.set_xlim(0, df_types['weight'].max() * 1.2)

# Revenue per unit
ax2 = axes[1]
df_types['rev_per_unit'] = (df_types['revenue'] / df_types['weight']).round(2)
df_rpu = df_types.sort_values('rev_per_unit', ascending=True)
bars2 = ax2.barh(df_rpu['type'], df_rpu['rev_per_unit'],
                 color=COFFEE_PALETTE[:4], height=0.5, zorder=3, edgecolor='#0f0a06')
ax2.bar_label(bars2, labels=[f'${v:.2f}' for v in df_rpu['rev_per_unit']],
              padding=6, color=CREAM, fontsize=10)
ax2.set_title('Revenue per Unit Sold', fontsize=13)
ax2.grid(True, axis='x', alpha=0.4)
ax2.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v:.0f}'))
ax2.set_xlim(0, df_types['rev_per_unit'].max() * 1.25)

fig.suptitle('Volume vs Pricing Efficiency by Coffee Type',
             fontsize=14, color=CREAM, fontweight='bold')
plt.tight_layout()
plt.show()

**Insights ►**
- **Arabica sells the most units** (947) yet earns the second-lowest revenue per unit — it's a high-volume, lower-priced offering.
- **Excelsa and Liberica** strike the best balance: strong unit sales AND higher per-unit revenue.
- **Robusta** underperforms on both volume and price — reinforcing the earlier margin concern.

---
## 4 · Geographic Distribution

### 4a · Revenue by Country

This business serves three markets: the United States, Ireland, and the United Kingdom.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Donut
ax = axes[0]
country_colors = [GREEN, BLUE, GOLD2]
wedges, texts, autotexts = ax.pie(
    df_country['revenue'],
    labels=df_country['country'],
    colors=country_colors,
    autopct=lambda p: f'{p:.1f}%\n${p/100*df_country["revenue"].sum():,.0f}',
    startangle=200,
    wedgeprops=dict(width=0.55, edgecolor='#0f0a06', linewidth=2),
    pctdistance=0.75
)
for t in texts:      t.set_color(CREAM); t.set_fontweight('bold'); t.set_fontsize(10)
for at in autotexts: at.set_color('#1a1108'); at.set_fontweight('bold'); at.set_fontsize(8)
ax.text(0, 0, 'Market\nShare', ha='center', va='center', color=CREAM,
        fontsize=11, fontweight='bold')
ax.set_title('Revenue by Country', fontsize=13)

# Bar + share %
ax2 = axes[1]
df_c_s = df_country.sort_values('revenue', ascending=True)
total = df_country['revenue'].sum()
bars = ax2.barh(df_c_s['country'], df_c_s['revenue'],
                color=country_colors[::-1], height=0.45, zorder=3, edgecolor='#0f0a06')
ax2.bar_label(bars, labels=[f'${v:,.0f} ({v/total*100:.1f}%)' for v in df_c_s['revenue']],
              padding=6, color=CREAM, fontsize=10)
ax2.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v/1000:.0f}K'))
ax2.set_title('Revenue Amount by Country', fontsize=13)
ax2.grid(True, axis='x', alpha=0.4)
ax2.set_xlim(0, df_country['revenue'].max() * 1.38)

fig.suptitle('Geographic Revenue Distribution — 3 Countries',
             fontsize=14, color=CREAM, fontweight='bold')
plt.tight_layout()
plt.show()

**Insights ►**
- The **United States drives 79% of all revenue** (\$35,639) — by a massive margin.
- **Ireland contributes 14.8%** (\$6,697) — disproportionately large relative to population, suggesting a loyal niche customer base or a key business account.
- **UK at 6.2%** (\$2,799) is the weakest market and may represent an untapped growth opportunity, especially given coffee culture trends in the UK.

### 4b · Top 10 Cities by Revenue

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6))

df_city_s = df_city.sort_values('revenue', ascending=True)
# Gradient colors by rank
n = len(df_city_s)
city_colors = [f'hsl({35 + i*8}, 70%, {55 - i*2}%)' for i in range(n)]
# Convert to hex for matplotlib
import colorsys
city_colors_mpl = [
    colorsys.hls_to_rgb((35 + i*8)/360, (55 - i*2)/100, 0.70)
    for i in range(n)
]

bars = ax.barh(df_city_s['city'], df_city_s['revenue'],
               color=city_colors_mpl, height=0.6, zorder=3, edgecolor='#0f0a06')

# Revenue labels
ax.bar_label(bars, labels=[f'${v:,.0f}' for v in df_city_s['revenue']],
             padding=6, color=CREAM, fontsize=10)

# Highlight top city
max_city = df_city_s.iloc[-1]
ax.annotate('🏆 Top City', xy=(max_city['revenue'], n-1),
            xytext=(max_city['revenue'] - 300, n - 1.6),
            arrowprops=dict(arrowstyle='->', color=GOLD2, lw=1.5),
            color=GOLD2, fontsize=9, fontweight='bold')

ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v:,.0f}'))
ax.set_title('Top 10 Cities by Revenue', fontsize=14, color=CREAM, pad=14)
ax.set_xlabel('Revenue (USD)')
ax.grid(True, axis='x', alpha=0.4)
ax.set_xlim(0, df_city['revenue'].max() * 1.3)

plt.tight_layout()
plt.show()

**Insights ►**
- **Washington D.C. leads** all US cities at \$1,067 — almost 30% more than Houston in second place.
- The **top 10 cities account for roughly \$6,540** of the total \$35,639 US revenue — just 18%. Revenue is **geographically distributed**, not concentrated in a few hubs.
- **Toledo appearing in top 3** is surprising — it may indicate a business account (café/restaurant) rather than individual consumers.

---
## 5 · Customer Intelligence

### 5a · Top 10 Customers by Revenue

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6))

df_cust_s = df_customers.sort_values('revenue', ascending=True)
max_rev = df_cust_s['revenue'].max()

# Bar
bars = ax.barh(df_cust_s['name'], df_cust_s['revenue'],
               color=[GOLD2 if i == len(df_cust_s)-1 else
                      '#c0c0c0' if i == len(df_cust_s)-2 else
                      '#d4845a' if i == len(df_cust_s)-3 else AMBER
                      for i in range(len(df_cust_s))],
               height=0.55, zorder=3, edgecolor='#0f0a06')

# Labels: revenue + orders
for i, (_, row) in enumerate(df_cust_s.iterrows()):
    ax.text(row['revenue'] + 8, i, f"${row['revenue']:.2f}  ({row['orders']} orders)",
            va='center', color=CREAM, fontsize=9)

# Medal annotations
medals = {len(df_cust_s)-1: '🥇', len(df_cust_s)-2: '🥈', len(df_cust_s)-3: '🥉'}
for i, medal in medals.items():
    ax.text(-12, i, medal, va='center', fontsize=12)

ax.set_xlim(-20, max_rev * 1.45)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v:.0f}'))
ax.set_title('Top 10 Customers by Total Revenue Spent', fontsize=14, color=CREAM, pad=14)
ax.set_xlabel('Revenue (USD)')
ax.grid(True, axis='x', alpha=0.4)

plt.tight_layout()
plt.show()

**Insights ►**
- **Allis Wilmore** leads at \$317 across just 3 orders — a high-value customer with a high average order value (~\$106).
- **Brenn Dundredge** at \$307 made 4 orders — the most frequent buyer in the top 10.
- **Nealson Cuttler** spent \$281.68 in a **single order** — the largest single transaction in the top 10.
- The top 10 customers represent ~\$2,601 combined, which is only **5.8% of total revenue** — the customer base is broad, not top-heavy. No single customer poses a dependency risk.

### 5b · Loyalty Card Analysis

Does holding a loyalty card actually change purchasing behaviour?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5.5))

# 1. Revenue split donut
ax = axes[0]
loyalty_colors = [RED, GREEN]
wedges, texts, autotexts = ax.pie(
    df_loyalty['revenue'],
    labels=df_loyalty['status'],
    colors=loyalty_colors,
    autopct=lambda p: f'{p:.1f}%\n${p/100*df_loyalty["revenue"].sum():,.0f}',
    startangle=90,
    wedgeprops=dict(width=0.55, edgecolor='#0f0a06', linewidth=2),
    pctdistance=0.75
)
for t in texts:      t.set_color(CREAM); t.set_fontweight('bold')
for at in autotexts: at.set_color('#1a1108'); at.set_fontweight('bold'); at.set_fontsize(9)
ax.text(0, 0, 'Revenue\nSplit', ha='center', va='center', color=CREAM, fontsize=10, fontweight='bold')
ax.set_title('Revenue by Loyalty Status', fontsize=13)

# 2. Customer count comparison
ax2 = axes[1]
bars = ax2.bar(df_loyalty['status'], df_loyalty['customers'],
               color=loyalty_colors, width=0.45, zorder=3, edgecolor='#0f0a06')
ax2.bar_label(bars, labels=[f'{v:,}\ncustomers' for v in df_loyalty['customers']],
              padding=6, color=CREAM, fontsize=11, fontweight='bold')
ax2.set_title('Customer Count', fontsize=13)
ax2.grid(True, axis='y', alpha=0.4)
ax2.set_ylim(0, df_loyalty['customers'].max() * 1.3)

# 3. Average order value comparison
ax3 = axes[2]
bars3 = ax3.bar(df_loyalty['status'], df_loyalty['avg_order'],
                color=loyalty_colors, width=0.45, zorder=3, edgecolor='#0f0a06')
ax3.bar_label(bars3, labels=[f'${v:.2f}' for v in df_loyalty['avg_order']],
              padding=6, color=CREAM, fontsize=12, fontweight='bold')
ax3.set_title('Average Order Value ($)', fontsize=13)
ax3.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v:.0f}'))
ax3.grid(True, axis='y', alpha=0.4)
ax3.set_ylim(0, df_loyalty['avg_order'].max() * 1.3)

# Annotate the difference
diff = df_loyalty['avg_order'].iloc[0] - df_loyalty['avg_order'].iloc[1]
ax3.annotate(f'Non-loyal spend\n${diff:.2f} more per order',
             xy=(0.5, df_loyalty['avg_order'].mean()),
             ha='center', color=MUTED, fontsize=9, style='italic')

fig.suptitle('Loyalty Card — Revenue, Reach & Spending Behaviour',
             fontsize=14, color=CREAM, fontweight='bold')
plt.tight_layout()
plt.show()

**Insights ►**
- **Non-loyalty card holders generate more revenue** (\$24,216 vs \$20,918) despite being only slightly more numerous (470 vs 443).
- **Non-loyal customers also spend more per order** (\$46.48 vs \$43.67) — a counter-intuitive finding that challenges the assumption that loyalty cards drive higher spend.
- The loyalty programme may need a redesign: it isn't yet translating into the higher order values it should incentivise. **Tiered rewards or volume discounts** could help.

---
## 6 · Radar Overview — Multi-Metric Bean Comparison

Let's pull together multiple dimensions — revenue, profit, margin, and volume — into a single radar chart for each coffee type.

In [ ]:
from matplotlib.patches import FancyArrowPatch

categories = ['Revenue\nShare', 'Profit\nShare', 'Margin\n%', 'Volume\nShare']
N = len(categories)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]  # close the polygon

# Normalise each metric to 0-100
rev_total    = df_types['revenue'].sum()
profit_total = df_types['profit'].sum()
vol_total    = df_types['weight'].sum()
max_margin   = df_types['margin'].max()

fig, axes = plt.subplots(2, 2, figsize=(13, 10), subplot_kw=dict(polar=True))
fig.patch.set_facecolor(BG)
axes = axes.flatten()

for idx, (_, row) in enumerate(df_types.iterrows()):
    values = [
        row['revenue'] / rev_total * 100,
        row['profit']  / profit_total * 100,
        row['margin']  / max_margin * 100,
        row['weight']  / vol_total * 100,
    ]
    values += values[:1]

    color = COFFEE_PALETTE[idx]
    ax = axes[idx]
    ax.set_facecolor('#1e1509')
    ax.spines['polar'].set_color('#3d2e1a')

    ax.plot(angles, values, color=color, linewidth=2.5, zorder=3)
    ax.fill(angles, values, color=color, alpha=0.25, zorder=2)
    ax.scatter(angles[:-1], values[:-1], color=color, s=60, zorder=4, edgecolors='#1a1108', linewidths=1.5)

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories, color=CREAM, fontsize=9)
    ax.set_yticks([25, 50, 75, 100])
    ax.set_yticklabels(['25%','50%','75%','100%'], color=MUTED, fontsize=7)
    ax.tick_params(colors=MUTED)
    ax.grid(color='#3d2e1a', linewidth=0.8)
    ax.set_ylim(0, 100)
    ax.set_title(f'{row["type"]}', color=color, fontsize=14, fontweight='bold', pad=18)

fig.suptitle('Coffee Type — Multi-Metric Radar Comparison\n(normalised to category max)',
             fontsize=14, color=CREAM, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

**Insights ►**
- **Liberica** has the most balanced profile — strong margin, decent volume, and healthy profit share. The ideal "star" product.
- **Excelsa** excels on revenue and profit share but lacks margin depth.
- **Arabica** leads on volume but lags on margin — classic high-turnover, low-profit product.
- **Robusta** is consistently weak across all metrics — the clearest candidate for pricing or promotional intervention.

---
## 7 · Interactive Exploration with Plotly

### 7a · Interactive Revenue Timeline

In [ ]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_monthly['period'],
    y=df_monthly['revenue'],
    mode='lines+markers',
    name='Monthly Revenue',
    line=dict(color=GOLD2, width=2.5),
    marker=dict(size=6, color=GOLD2, line=dict(color='#1a1108', width=1.5)),
    fill='tozeroy',
    fillcolor='rgba(240,184,96,0.12)',
    hovertemplate='<b>%{x}</b><br>Revenue: $%{y:,.2f}<extra></extra>'
))

# Rolling average
rolling = df_monthly['revenue'].rolling(3, center=True).mean()
fig.add_trace(go.Scatter(
    x=df_monthly['period'],
    y=rolling,
    mode='lines',
    name='3-Month Moving Avg',
    line=dict(color=AMBER, width=2, dash='dot'),
    hovertemplate='<b>%{x}</b><br>3M Avg: $%{y:,.2f}<extra></extra>'
))

# Year bands
year_colors = ['rgba(212,148,58,0.05)', 'rgba(232,117,42,0.05)',
               'rgba(76,175,125,0.05)', 'rgba(91,155,213,0.05)']
year_ranges = [('2019-01','2019-12'), ('2020-01','2020-12'),
               ('2021-01','2021-12'), ('2022-01','2022-08')]
for (x0, x1), col, yr in zip(year_ranges, year_colors, [2019,2020,2021,2022]):
    fig.add_vrect(x0=x0, x1=x1, fillcolor=col, layer='below', line_width=0,
                  annotation_text=str(yr), annotation_position='top left',
                  annotation_font_color=MUTED, annotation_font_size=11)

fig.update_layout(
    title=dict(text='☕ Monthly Revenue & 3-Month Moving Average (2019–2022)',
               font=dict(size=16, color=CREAM)),
    paper_bgcolor=BG, plot_bgcolor=CARD,
    font=dict(color=MUTED),
    xaxis=dict(title='Month', gridcolor='#3d2e1a', showline=False,
               tickangle=-45, nticks=15),
    yaxis=dict(title='Revenue (USD)', gridcolor='#3d2e1a',
               tickformat='$,.0f'),
    legend=dict(bgcolor='#1a1108', bordercolor='#3d2e1a', borderwidth=1,
                font=dict(color=CREAM)),
    hovermode='x unified',
    height=480
)

fig.show()

### 7b · Interactive Product Scatter — Revenue vs Profit Margin

In [ ]:
fig = go.Figure()

for i, (_, row) in enumerate(df_types.iterrows()):
    fig.add_trace(go.Scatter(
        x=[row['revenue']],
        y=[row['margin']],
        mode='markers+text',
        name=row['type'],
        marker=dict(size=row['weight']/20, color=COFFEE_PALETTE[i],
                    line=dict(color='#1a1108', width=2), opacity=0.85),
        text=[row['type']],
        textposition='top center',
        textfont=dict(color=CREAM, size=11),
        hovertemplate=(
            f"<b>{row['type']}</b><br>"
            f"Revenue: ${row['revenue']:,.0f}<br>"
            f"Margin: {row['margin']}%<br>"
            f"Profit: ${row['profit']:,.0f}<br>"
            f"Units: {row['weight']}<extra></extra>"
        )
    ))

# Quadrant lines
fig.add_hline(y=10, line_dash='dot', line_color=MUTED, opacity=0.5,
              annotation_text='Avg Margin 10%', annotation_font_color=MUTED)
fig.add_vline(x=df_types['revenue'].mean(), line_dash='dot', line_color=MUTED, opacity=0.5,
              annotation_text='Avg Revenue', annotation_font_color=MUTED)

fig.update_layout(
    title=dict(text='Product Positioning — Revenue vs Margin (bubble size = units sold)',
               font=dict(size=16, color=CREAM)),
    paper_bgcolor=BG, plot_bgcolor=CARD,
    font=dict(color=MUTED),
    xaxis=dict(title='Total Revenue (USD)', gridcolor='#3d2e1a', tickformat='$,.0f'),
    yaxis=dict(title='Profit Margin (%)', gridcolor='#3d2e1a', ticksuffix='%'),
    legend=dict(bgcolor='#1a1108', bordercolor='#3d2e1a', borderwidth=1,
                font=dict(color=CREAM)),
    height=480
)
fig.show()

### 7c · Interactive Country & City Breakdown

In [ ]:
fig = make_subplots(
    rows=1, cols=2,
    specs=[[{'type': 'pie'}, {'type': 'bar'}]],
    subplot_titles=['Revenue by Country', 'Top 10 Cities']
)

fig.add_trace(go.Pie(
    labels=df_country['country'],
    values=df_country['revenue'],
    hole=0.55,
    marker=dict(colors=[GREEN, BLUE, GOLD2], line=dict(color='#1a1108', width=2)),
    textinfo='label+percent',
    hovertemplate='<b>%{label}</b><br>Revenue: $%{value:,.2f}<br>Share: %{percent}<extra></extra>'
), row=1, col=1)

fig.add_trace(go.Bar(
    x=df_city['revenue'],
    y=df_city['city'],
    orientation='h',
    marker=dict(
        color=df_city['revenue'],
        colorscale=[[0, '#5a3010'], [0.5, AMBER], [1, GOLD2]],
        showscale=False
    ),
    hovertemplate='<b>%{y}</b><br>Revenue: $%{x:,.2f}<extra></extra>'
), row=1, col=2)

fig.update_layout(
    title=dict(text='Geographic Revenue Distribution', font=dict(size=16, color=CREAM)),
    paper_bgcolor=BG, plot_bgcolor=CARD,
    font=dict(color=MUTED),
    showlegend=False,
    height=500
)
fig.update_xaxes(gridcolor='#3d2e1a', tickformat='$,.0f', row=1, col=2)
fig.update_yaxes(gridcolor='#3d2e1a', row=1, col=2)

for ann in fig['layout']['annotations']:
    ann['font'] = dict(size=13, color=CREAM)

fig.show()

---
## 8 · Summary & Strategic Recommendations

Let's bring the story together into a final summary dashboard.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.patch.set_facecolor(BG)

# 1 — Revenue by Type
ax = axes[0, 0]
ax.bar(df_types['type'], df_types['revenue'], color=COFFEE_PALETTE[:4],
       zorder=3, edgecolor='#0f0a06', width=0.55)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v/1000:.0f}K'))
ax.set_title('Revenue by Bean Type', fontsize=12)
ax.grid(True, axis='y', alpha=0.4)

# 2 — Roast preference
ax = axes[0, 1]
wedges, texts, ats = ax.pie(df_roast['revenue'], labels=df_roast['roast'],
    colors=[GOLD2, GOLD, AMBER], autopct='%1.0f%%', startangle=90,
    wedgeprops=dict(edgecolor='#0f0a06', linewidth=2), pctdistance=0.78)
for t in texts: t.set_color(CREAM); t.set_fontsize(9)
for at in ats:  at.set_color('#1a1108'); at.set_fontweight('bold'); at.set_fontsize(8)
ax.set_title('Roast Preference', fontsize=12)

# 3 — Pack size revenue
ax = axes[0, 2]
ax.bar(df_size['size'], df_size['revenue'], color=[MUTED, AMBER, GOLD, GOLD2],
       zorder=3, edgecolor='#0f0a06', width=0.55)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v/1000:.0f}K'))
ax.set_title('Revenue by Pack Size', fontsize=12)
ax.grid(True, axis='y', alpha=0.4)

# 4 — Country
ax = axes[1, 0]
bars = ax.barh(df_country['country'], df_country['revenue'],
               color=[GREEN, BLUE, GOLD2][::-1], height=0.45, zorder=3, edgecolor='#0f0a06')
ax.bar_label(bars, labels=[f'${v:,.0f}' for v in df_country['revenue'][::-1]],
             padding=5, color=CREAM, fontsize=9)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v/1000:.0f}K'))
ax.set_title('Revenue by Country', fontsize=12)
ax.grid(True, axis='x', alpha=0.4)

# 5 — Loyalty
ax = axes[1, 1]
ax.bar(df_loyalty['status'], df_loyalty['avg_order'], color=[RED, GREEN],
       zorder=3, edgecolor='#0f0a06', width=0.45)
ax.bar_label(ax.containers[0], labels=[f'${v:.2f}' for v in df_loyalty['avg_order']],
             padding=5, color=CREAM, fontsize=11, fontweight='bold')
ax.set_title('Avg Order Value: Loyalty vs Non', fontsize=12)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v:.0f}'))
ax.grid(True, axis='y', alpha=0.4)
ax.set_ylim(0, 60)

# 6 — Margin radar summary as bar
ax = axes[1, 2]
bar_colors2 = [GREEN if m >= 10 else AMBER if m >= 8 else RED for m in df_types['margin']]
bars6 = ax.bar(df_types['type'], df_types['margin'], color=bar_colors2,
               zorder=3, edgecolor='#0f0a06', width=0.55)
ax.bar_label(bars6, labels=[f'{v}%' for v in df_types['margin']],
             padding=5, color=CREAM, fontsize=11, fontweight='bold')
ax.axhline(10, color=MUTED, linewidth=1.2, linestyle='--', alpha=0.7)
ax.set_title('Profit Margin by Bean Type', fontsize=12)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:.0f}%'))
ax.grid(True, axis='y', alpha=0.4)
ax.set_ylim(0, 18)

for ax in axes.flatten():
    ax.set_facecolor(CARD)
    for spine in ax.spines.values():
        spine.set_edgecolor('#3d2e1a')

fig.suptitle('☕ Coffee Bean Sales — Full Story Summary Dashboard',
             fontsize=17, color=CREAM, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

---
## 9 · Final Findings & Strategic Recommendations

| # | Area | Finding | Recommendation |
|---|------|---------|----------------|
| 1 | **Revenue trend** | 2021 was the best year (+13.6% YoY). 2022 appears to decline but data is truncated at Aug. | Monitor full-year 2022 figures before drawing conclusions. |
| 2 | **Best bean** | Liberica has the highest margin (13%) despite not leading on revenue. | Promote Liberica in upsell campaigns and bundles. |
| 3 | **Worst bean** | Robusta lags across all metrics — lowest revenue, lowest margin (6%). | Consider a price increase or limited promotional offer to test demand elasticity. |
| 4 | **Pack size** | 2.5 kg packs generate 53% of revenue. Margins are equal across sizes. | Incentivise bulk buying via small discounts on 2.5 kg; bundle 0.2 kg as free trial. |
| 5 | **Roast** | Light roast is most popular (38.5%). Dark roast is least popular (29.2%). | Lead with light roast in marketing. Consider dark roast espresso blend partnerships. |
| 6 | **Geography** | US drives 79% of revenue. Ireland punches above its weight. UK lags. | Investigate what drives Ireland success and replicate in the UK. |
| 7 | **Loyalty** | Loyalty card holders spend *less* per order (\$43.67 vs \$46.48). | Redesign loyalty programme: introduce minimum spend tiers or bonus points above \$50. |
| 8 | **Customers** | Top 10 customers = only 5.8% of revenue. No single dependency risk. | This is healthy. Focus on growing mid-tier customers into repeat buyers. |

---

> *☕ End of Story — Coffee Bean Sales Intelligence Dashboard | Data: 2019–2022 | Built with pandas, numpy, matplotlib, seaborn & plotly*